- 데이터: `../results/emnist`에 이미 캐시된 EMNIST ByClass를 그대로 재사용하되(재다운로드 없음),
  거기서 소문자를 제외시켜서 대문자+숫자인 총 36 class 사용
- hyperparameters: batch 128, epoch 10, Adam lr=0.001(betas 기본값), seed 261014
- 목표: XC7Z020 BRAM 630KB(645,120B) 예산 안에 들어오는 조합을 정확도와 함께 비교
- 실행 위치: `tb`에서 커널을 띄운다 (`lightletter-tb`/venv 커널 선택)

In [1]:
import math
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.datasets import EMNIST

DATA_ROOT = Path('../results/emnist')
BRAM_BUDGET_BYTES = 630 * 1024  # XC7Z020, weight-only 하한과 비교하는 예산
NUM_CLASSES = 36
SEED = 261014
BATCH_SIZE = 128
EPOCHS = 10
LR = .001
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE)


def load_split(partition):
    dataset = EMNIST(root=str(DATA_ROOT), split='byclass', train=(partition == 'train'), download=True)
    images = dataset.data.numpy()
    labels = dataset.targets.numpy()
    images = np.transpose(images, (0, 2, 1))  # EMNIST 회전+반전 아티팩트 보정
    keep = labels < NUM_CLASSES
    return images[keep].copy(), labels[keep].astype(np.int64).copy()


x_train, y_train = load_split('train')
x_test, y_test = load_split('test')
print('train:', x_train.shape, 'test:', x_test.shape)

device: mps


train: (533993, 28, 28) test: (89264, 28, 28)


## 파라미터화한 모델

`model.py`의 `Net`/`Quant16`을 그대로 옮기되, `conv_channels`·`padding`·
`pool_stride`·`fc_widths`를 생성자 인자로 받게 해서 조합을 빠르게 바꿔볼 수 있게 한다.
kernel(3x3)·pool kernel(2x2)·activation(ReLU)·양자화(INT16 QAT)는 과제 제약대로 고정.

In [2]:
class Quant16(nn.Module):
    """대칭 signed INT16 fake-quantization. tb/cnn_golden/model.py와 동일."""
    def __init__(self, weight=False):
        super().__init__()
        self.weight = weight
        self.enabled = True
        self.register_buffer('maximum', torch.tensor(0.))

    @property
    def scale(self):
        return 2. ** torch.ceil(torch.log2(self.maximum.clamp_min(1e-12) / 32767.))

    def forward(self, x):
        if not self.enabled:
            return x
        if self.training:
            with torch.no_grad():
                maximum = x.detach().abs().amax()
                self.maximum.copy_(maximum if self.weight else torch.maximum(self.maximum, maximum))
        scale = self.scale.detach()
        clipped = (x / scale).clamp(-32768, 32767)
        rounded = clipped + (clipped.round() - clipped).detach()
        return rounded * scale


class Net(nn.Module):
    def __init__(self, conv_channels, padding, pool_stride, fc_widths):
        super().__init__()
        self.conv_channels = conv_channels
        self.padding = padding
        self.pool_stride = pool_stride
        self.convs = nn.ModuleList([
            nn.Conv2d(conv_channels[i], conv_channels[i + 1], 3, padding=padding)
            for i in range(len(conv_channels) - 1)
        ])
        size = 28
        self.sizes = []
        for _ in self.convs:
            size = size - 2 + 2 * padding
            size = (size - 2) // pool_stride + 1
            if size < 1:
                raise ValueError('Empty spatial output')
            self.sizes.append(size)
        widths = [conv_channels[-1] * size * size] + list(fc_widths)
        self.fcs = nn.ModuleList([nn.Linear(a, b) for a, b in zip(widths, widths[1:])])
        self.input_quant = Quant16()
        self.weight_quant = nn.ModuleList([Quant16(weight=True) for _ in [*self.convs, *self.fcs]])
        self.activation_quant = nn.ModuleList([Quant16() for _ in [*self.convs, *self.fcs]])

    def quantization(self, enabled):
        for module in self.modules():
            if isinstance(module, Quant16):
                module.enabled = enabled

    def forward(self, x):
        x = self.input_quant(x)
        for i, conv in enumerate(self.convs):
            x = F.conv2d(x, self.weight_quant[i](conv.weight), conv.bias, padding=self.padding)
            x = self.activation_quant[i](F.relu(x))
            x = F.max_pool2d(x, 2, self.pool_stride)
        x = x.flatten(1)
        for j, fc in enumerate(self.fcs):
            i = len(self.convs) + j
            x = F.linear(x, self.weight_quant[i](fc.weight), fc.bias)
            if j < len(self.fcs) - 1:
                x = F.relu(x)
            x = self.activation_quant[i](x)
        return x

    def inventory(self):
        weights = sum(m.weight.numel() for m in [*self.convs, *self.fcs])
        biases = sum(m.bias.numel() for m in [*self.convs, *self.fcs])
        weight_bytes = weights * 2
        return dict(weights=weights, biases=biases, weight_int16_bytes=weight_bytes,
                    bram_ratio=weight_bytes / BRAM_BUDGET_BYTES,
                    spatial_outputs=self.sizes)


## 학습 없이 크기만 확인

실제로 학습을 돌리기 전에, 후보 조합들의 가중치 용량과 BRAM 예산 대비 배율을 먼저 계산한다.
지금까지 대화에서 손으로 계산했던 값들을 그대로 재현해서 맞는지 검증한다.

In [3]:
candidates = {
    '1-6-16, padding=1 (현재 확정안)': dict(conv_channels=[1, 6, 16], padding=1, pool_stride=1, fc_widths=[256, 64, 36]),
    '1-6-16, padding=0': dict(conv_channels=[1, 6, 16], padding=0, pool_stride=1, fc_widths=[256, 64, 36]),
    '1-6-16, padding=1, FC1=128': dict(conv_channels=[1, 6, 16], padding=1, pool_stride=1, fc_widths=[128, 64, 36]),
}

print(f"{'구성':35s} {'flatten':>10s} {'weights':>12s} {'bytes':>12s} {'BRAM배율':>10s}")
for name, cfg in candidates.items():
    net = Net(**cfg)
    inv = net.inventory()
    flatten = cfg['conv_channels'][-1] * inv['spatial_outputs'][-1] ** 2
    print(f"{name:35s} {flatten:10d} {inv['weights']:12d} {inv['weight_int16_bytes']:12d} {inv['bram_ratio']:9.2f}x")


구성                                     flatten      weights        bytes     BRAM배율
1-6-16, padding=1 (현재 확정안)               10816      2788502      5577004      8.64x
1-6-16, padding=0                         7744      2002070      4004140      6.21x
1-6-16, padding=1, FC1=128               10816      1395862      2791724      4.33x


## 실제 학습 — 정확도까지 실측

크기 계산만으로는 정확도를 알 수 없으니, 후보 조합을 실제로 10 epoch 학습해서 확인한다.
hyperparameter·seed·class weight는 `tb/cnn_golden/train.py`와 동일하게 맞춘다.

In [4]:
def batch(x, ids, device):
    return torch.from_numpy(np.asarray(x[ids, None], dtype=np.float32) / 255.).to(device)


@torch.no_grad()
def predict(net, x, device):
    net.eval()
    result = []
    for offset in range(0, len(x), BATCH_SIZE):
        result.append(net(batch(x, slice(offset, offset + BATCH_SIZE), device)).argmax(1).cpu().numpy())
    return np.concatenate(result)


def scores(labels, predictions):
    cm = np.bincount(labels * 36 + predictions, minlength=36 * 36).reshape(36, 36)
    return dict(accuracy=float(np.mean(labels == predictions)),
                digit_accuracy=float(np.mean(labels[labels < 10] == predictions[labels < 10])),
                letter_accuracy=float(np.mean(labels[labels >= 10] == predictions[labels >= 10])),
                balanced_accuracy=float(np.mean(cm.diagonal() / cm.sum(1))))


def train_and_evaluate(conv_channels, padding, pool_stride, fc_widths, device=DEVICE):
    counts = np.bincount(y_train, minlength=36)
    letter_weight = float(counts[:10].sum() / counts[10:].sum())
    class_weights = torch.ones(36, device=device)
    class_weights[10:] = letter_weight

    torch.manual_seed(SEED)
    net = Net(conv_channels, padding, pool_stride, fc_widths).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        net.train()
        order = np.random.default_rng(SEED + epoch).permutation(len(y_train))
        total = 0.
        for offset in range(0, len(y_train), BATCH_SIZE):
            ids = order[offset:offset + BATCH_SIZE]
            target = torch.from_numpy(np.array(y_train[ids])).to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(net(batch(x_train, ids, device)), target, weight=class_weights)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(ids)
        print(f'epoch {epoch + 1}/{EPOCHS} loss={total / len(y_train):.4f}')

    quantized = predict(net, x_test, device)
    return net, scores(y_test, quantized)


In [5]:
# padding=0으로 바꾼 안 (채널 1->6->16, FC 256->64->36) 실측
net_pad0, result_pad0 = train_and_evaluate(conv_channels=[1, 6, 16], padding=0, pool_stride=1,
                                            fc_widths=[256, 64, 36])
inv_pad0 = net_pad0.inventory()
print()
print('spatial_outputs:', inv_pad0['spatial_outputs'])
print('weights:', inv_pad0['weights'], 'bytes:', inv_pad0['weight_int16_bytes'],
      f"BRAM 배율: {inv_pad0['bram_ratio']:.2f}x")
print('accuracy:', result_pad0)


epoch 1/10 loss=0.3836


epoch 2/10 loss=0.2345


epoch 3/10 loss=0.2091


epoch 4/10 loss=0.1939


epoch 5/10 loss=0.1818


...

epoch 7/10 loss=0.1644


epoch 8/10 loss=0.1580


epoch 9/10 loss=0.1516


epoch 10/10 loss=0.1465



spatial_outputs: [25, 22]
weights: 2002070 bytes: 4004140 BRAM 배율: 6.21x
accuracy: {'accuracy': 0.92336216167772, 'digit_accuracy': 0.9413135812700715, 'letter_accuracy': 0.890193326102214, 'balanced_accuracy': 0.9214265416582855}


## 결과 비교

| 구성 | flatten | weights | bytes | BRAM 배율 | 전체 정확도 | 숫자 정확도 | 문자 정확도 | Balanced |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 1-6-16, padding=1 (현재 확정안) | 10,816 | 2,788,502 | 5,577,004 | 8.64x | 92.27% | 93.89% | 89.28% | 92.19% |
| **1-6-16, padding=0 (실측)** | 7,744 | 2,002,070 | 4,004,140 | **6.21x** | 92.34% | 94.13% | 89.02% | 92.14% |

패딩을 없애도 정확도는 사실상 동일하다(전체 +0.07%p, balanced -0.05%p — 노이즈 수준).
반면 BRAM 배율은 8.64x -> 6.21x로 약 28% 줄었다. 즉 패딩 제거는 정확도 손실 없이 순수 이득이었다.

다만 여전히 예산을 6.21배 초과하므로 이것만으로는 부족하다. FC1 폭을 128로 줄이면(패딩=1 기준
4.33x였음) 패딩=0과 결합했을 때 더 줄어들 것으로 예상되며, 다음 탐색 후보다.

In [6]:
# padding=0 + FC1=128로 줄인 안 (채널 1->6->16) 실측 — pad0와 FC128 개선을 합친 다음 후보
net_pad0_fc128, result_pad0_fc128 = train_and_evaluate(conv_channels=[1, 6, 16], padding=0, pool_stride=1,
                                                         fc_widths=[128, 64, 36])
inv_pad0_fc128 = net_pad0_fc128.inventory()
print()
print('spatial_outputs:', inv_pad0_fc128['spatial_outputs'])
print('weights:', inv_pad0_fc128['weights'], 'bytes:', inv_pad0_fc128['weight_int16_bytes'],
      f"BRAM 배율: {inv_pad0_fc128['bram_ratio']:.2f}x")
print('accuracy:', result_pad0_fc128)


epoch 1/10 loss=0.4233


epoch 2/10 loss=0.2528


epoch 3/10 loss=0.2264


epoch 4/10 loss=0.2123


epoch 5/10 loss=0.2023


...

epoch 7/10 loss=0.1870


epoch 8/10 loss=0.1819


epoch 9/10 loss=0.1763


epoch 10/10 loss=0.1714



spatial_outputs: [25, 22]
weights: 1002646 bytes: 2005292 BRAM 배율: 3.11x
accuracy: {'accuracy': 0.9215921312063093, 'digit_accuracy': 0.9366690838772057, 'letter_accuracy': 0.8937344477764309, 'balanced_accuracy': 0.9200101217651253}


## 결과 비교 (FC1=128 추가)

| 구성 | flatten | weights | bytes | BRAM 배율 | 전체 정확도 | 숫자 정확도 | 문자 정확도 | Balanced |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 1-6-16, padding=1 (이전 확정안) | 10,816 | 2,788,502 | 5,577,004 | 8.64x | 92.27% | 93.89% | 89.28% | 92.19% |
| 1-6-16, padding=0 | 7,744 | 2,002,070 | 4,004,140 | 6.21x | 92.34% | 94.13% | 89.02% | 92.14% |
| **1-6-16, padding=0, FC1=128 (실측)** | 7,744 | 1,002,646 | 2,005,292 | **3.11x** | 92.16% | 93.67% | 89.37% | 92.00% |

FC1 폭을 256→128로 줄이면 weights가 거의 절반(2,002,070 → 1,002,646)으로 줄고 BRAM 배율도
6.21x → 3.11x로 거의 절반이 된다. 정확도는 전체 -0.18%p, balanced -0.14%p로 소폭 하락하지만
문자 정확도는 오히려 +0.35%p 올랐다 — 노이즈 범위 안에 가까운 변화다.

패딩 제거(-28%)와 FC1 축소(추가 -50%)를 합치면 원래 확정안(8.64x) 대비 BRAM을 약 64% 줄인
셈이다. 다만 예산(630KB)은 여전히 3.11배 초과한 상태라 이것만으로는 부족하다.

다음 후보: FC를 2단(256→64→36 계열) 대신 1단(64→36 또는 그보다 얇게)으로 더 줄이거나,
INT16 대신 INT8/INT4 가중치 양자화로 `bytes_per_weight` 자체를 낮추는 쪽을 검토한다.
